**Eye-Tracking & Neurocognitive Intelligence | Eye-Movement Modelling & Advanced AI | Presented by: Reza Saadatyar (2026)**

**04 | Prospective Main-Cohort GRU and Transformer Validation**

![Independent temporal model architectures and validation within source-participant partitions](../../images/04_gru_and_transformer_models.svg)

**Next main execution (not yet run on 100 people):** This notebook reads the current GazeBase Round 1 / Session 1 study with participant count configured once in Notebook 01 (100 by default, optionally 300 for a separate, frozen protocol). It classifies the FXS/HSS/RAN/TEX original task identity, **not** cognition or a clinical outcome.

The introductory figure and previous committed scores describe the **earlier ten-person pilot** and are not main-study results. Use the matching stage ZIPs in /content in any fresh Colab Runtime: explicitly uploaded archives take precedence over stale pilot files checked out from GitHub. Independent-person roles and the main-cohort count are checked. No Google Drive mount is involved. This notebook fits and scores Validation only; the main Test remains reserved.


**00. Colab Repository Setup**


In [ ]:
# ======================== 00. Colab Repository Setup =========================

# -------------------------- 1. Repository Checkout ---------------------------
from pathlib import Path
import subprocess

repo_url = "https://github.com/RezaSaadatyar/Eye-Tracking-Neurocognitive-Intelligence.git"
repo_path = Path("/content") / repo_url.rsplit("/", 1)[-1].removesuffix(".git")
if not repo_path.exists():
    subprocess.run(["git", "clone", "-q", repo_url, str(repo_path)], check=True)

# ---------------------------- 2. Open Repository -----------------------------
%cd {repo_path}


**01. Imports and Research Environment**


In [ ]:
# =================== 01. Imports and Research Environment ====================

# ------------------------ 1. Study Environment Setup -------------------------
from python_modules.core.colab_setup import setup_colab
root = setup_colab("eye_movement_modelling")

# -------------------- 2. Temporal Training and Evaluation --------------------
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import ConfusionMatrixDisplay, confusion_matrix
from IPython.display import display
from s2_handoff import ensure_s2_stage_inputs, verify_main_cohort
from core.output_download import download_outputs_from_colab
from core.plot_style import plot_style, save_figure

# ------------------------- 3. Study Model Utilities --------------------------
from s2_data import (
    TASKS, load_protocol, load_windows, verify_temporal_source_alignment,
)
from s2_training import fit_temporal
from s2_metrics import metric_row

# ----------------------------- 4. Output Folder ------------------------------
output_dir = root / "eye_movement_modelling" / "output"
output_dir.mkdir(parents=True, exist_ok=True)
plot_style(font_family="DejaVu Sans", figure_dpi=100)
print("Eye-Movement Modelling temporal outputs:", output_dir)


**02. Recover Main Temporal Inputs**

Use matching stage-01 windows, frozen protocol and independent-person Train/Validation/Test registry. A new ZIP overrides the previous pilot files in a fresh checkout; verify current count and source identities.

The next 100-person run has not yet produced results in this section; previous numerical results and the linked illustration are from the ten-person pilot.


In [ ]:
# ==================== 02. Recover Frozen Temporal Inputs =====================

# -------------------- 1. Restore Explicit Stage 01 Files ---------------------
ensure_s2_stage_inputs(output_dir, 1, (
    "01_sequence_windows.npz",
    "01_split_assignments.csv", "01_protocol.csv",
))
windows = load_windows(output_dir / "01_sequence_windows.npz")
split = pd.read_csv(
    output_dir / "01_split_assignments.csv",
    dtype={"record_id": str, "participant_id": str},
)
protocol = load_protocol(output_dir / "01_protocol.csv")
role_counts = verify_main_cohort(protocol, split)
print("Main cohort participant roles:", role_counts)
if windows["x"].shape[1:] != (
    protocol["window_samples"], len(protocol["temporal_channels"]),
):
    raise ValueError("Sequence data does not match the saved source protocol")

# -------------------------- 2. Display Data Support --------------------------
record_roles = dict(zip(split.record_id, split.split))
alignment_counts = verify_temporal_source_alignment(windows, split)
roles = np.array([record_roles[str(r)] for r in windows["record_id"]])
print("Verified window counts by participant partition:", alignment_counts)
validation_people = split.loc[
    split.split.eq("validation"), "participant_id",
].nunique()
print("Study mode:", protocol["purpose"])
print("Sequence tensor:", windows["x"].shape)
print("Train/Validation/Test window counts:")
display(pd.Series(roles).value_counts().rename("n_windows").to_frame())
print("Independent Validation participants:", validation_people)
reported = []


**03. Train GRU**

Fit GRU only on Train windows and choose its checkpoint on new Validation source records. Keep the existing architecture and eight-epoch maximum. Save parameter count, selected epoch, predictions and curves.

The next 100-person run has not yet produced results in this section; previous numerical results and the linked illustration are from the ten-person pilot.


In [ ]:
# =================== 03. Train Gated Recurrent Unit (GRU) ====================

# --------------------- 1. Fit on Train Participants Only ---------------------
model_name = "gru"
history, predictions, info = fit_temporal(
    model_name, windows, split, output_dir / f"04_gru.pt",
    seed=protocol["split_seed"], epochs=8, patience=3,
    batch_size=64,
)
history.to_csv(output_dir / f"04_gru_history.csv", index=False)
predictions.to_csv(
    output_dir / f"04_gru_validation_predictions.csv", index=False,
)
print("Model:", model_name)
print("Best epoch:", info["best_epoch"])
print("Trainable parameters:", info["parameters"])
display(history.tail(8))

# --------------------- 2. Report Record-Level Validation ---------------------
result = metric_row(
    predictions["target_index"], predictions["pred_index"],
    model=model_name, partition="validation",
    n_participants=validation_people,
)
result["parameters"] = int(info["parameters"])
result["best_epoch"] = int(info["best_epoch"])
reported.append(result)
display(pd.DataFrame([result]))

# ------------------ 3. Display Learning and Error Evidence -------------------
fig, axes = plt.subplots(1, 2, figsize=(11.5, 4.4))
axes[0].plot(history["epoch"], history["train_loss"],
             marker="o", label="Train loss")
axes[0].plot(history["epoch"], history["validation_window_loss"],
             marker="o", label="Validation loss")
axes[0].set(xlabel="Epoch", ylabel="Cross-entropy",
            title="Gated Recurrent Unit (GRU): Learning Curves")
axes[0].grid(alpha=0.2)
axes[0].legend()
matrix = confusion_matrix(
    predictions.target_index, predictions.pred_index,
    labels=range(len(TASKS)),
)
ConfusionMatrixDisplay(matrix, display_labels=TASKS).plot(
    ax=axes[1], colorbar=False, values_format="d",
)
axes[1].set_title("Gated Recurrent Unit (GRU): Validation by Record")
fig.tight_layout()
save_figure(fig, output_dir / f"04_gru_evaluation.png")
plt.show()


**04. Train Transformer**

Fit the compact Transformer independently using the same current-cohort representation and people. Select by record-level Validation Macro F1; attention is not a faithful causal explanation.

The next 100-person run has not yet produced results in this section; previous numerical results and the linked illustration are from the ten-person pilot.


In [ ]:
# =================== 04. Train Compact Transformer Encoder ===================

# --------------------- 1. Fit on Train Participants Only ---------------------
model_name = "transformer"
history, predictions, info = fit_temporal(
    model_name, windows, split, output_dir / f"04_transformer.pt",
    seed=protocol["split_seed"], epochs=8, patience=3,
    batch_size=64,
)
history.to_csv(output_dir / f"04_transformer_history.csv", index=False)
predictions.to_csv(
    output_dir / f"04_transformer_validation_predictions.csv", index=False,
)
print("Model:", model_name)
print("Best epoch:", info["best_epoch"])
print("Trainable parameters:", info["parameters"])
display(history.tail(8))

# --------------------- 2. Report Record-Level Validation ---------------------
result = metric_row(
    predictions["target_index"], predictions["pred_index"],
    model=model_name, partition="validation",
    n_participants=validation_people,
)
result["parameters"] = int(info["parameters"])
result["best_epoch"] = int(info["best_epoch"])
reported.append(result)
display(pd.DataFrame([result]))

# ------------------ 3. Display Learning and Error Evidence -------------------
fig, axes = plt.subplots(1, 2, figsize=(11.5, 4.4))
axes[0].plot(history["epoch"], history["train_loss"],
             marker="o", label="Train loss")
axes[0].plot(history["epoch"], history["validation_window_loss"],
             marker="o", label="Validation loss")
axes[0].set(xlabel="Epoch", ylabel="Cross-entropy",
            title="Compact Transformer Encoder: Learning Curves")
axes[0].grid(alpha=0.2)
axes[0].legend()
matrix = confusion_matrix(
    predictions.target_index, predictions.pred_index,
    labels=range(len(TASKS)),
)
ConfusionMatrixDisplay(matrix, display_labels=TASKS).plot(
    ax=axes[1], colorbar=False, values_format="d",
)
axes[1].set_title("Compact Transformer Encoder: Validation by Record")
fig.tight_layout()
save_figure(fig, output_dir / f"04_transformer_evaluation.png")
plt.show()


**05. Compare Current Validation**

Save both neural models' record-level Validation metrics with model parameter counts and selected epochs. Do not use Test for training or selection.

The next 100-person run has not yet produced results in this section; previous numerical results and the linked illustration are from the ten-person pilot.


In [ ]:
# ====================== 05. Compare Temporal Validation ======================

# --------------------- 1. Save Shared Validation Metrics ---------------------
metrics = pd.DataFrame(reported).sort_values(
    ["macro_f1", "balanced_accuracy", "model"],
    ascending=[False, False, True],
)
metrics.to_csv(
    output_dir / "04_neural_validation_metrics.csv", index=False,
)
display(metrics)

# --------------------- 2. Plot Independent Model Scores ----------------------
fig, ax = plt.subplots(figsize=(7.8, 4.5))
pos = np.arange(len(metrics))
ax.bar(pos - 0.16, metrics["macro_f1"], 0.32, label="Macro F1")
ax.bar(pos + 0.16, metrics["balanced_accuracy"], 0.32,
       label="Balanced accuracy")
ax.set_xticks(pos, metrics["model"])
ax.set(ylabel="Record-level Validation score", ylim=(0, 1),
       title="Gated Recurrent Unit (GRU) and Compact Transformer Encoder: Validation Comparison")
ax.legend()
ax.grid(axis="y", alpha=0.2)
fig.tight_layout()
save_figure(fig, output_dir / "04_model_comparison.png")
plt.show()


**06. Stage-04 Interpretation and ZIP**

Package new GRU/Transformer checkpoints, histories, Validation predictions and figures for Notebook 05. Do not add output files larger than the Git per-blob limit.

The next 100-person run has not yet produced results in this section; previous numerical results and the linked illustration are from the ten-person pilot.


In [ ]:
# ======================= 06. Export Stage-04 Evidence ========================
download_outputs_from_colab("eye_movement_modelling/output", "04")
